In [4]:
print("Hello World")

Hello World


## Data Loading and Preprocessing

In [5]:

from sentence_splitter import sentence_splitter
import pandas as pd

ss = sentence_splitter()
df = pd.read_csv(r"datasets\annotations_raw.csv")

In [6]:
import pandas as pd

labeled_df = df[["doc_name","label", "text"]].dropna().reset_index(drop=True)

# Step 1: Combine texts within each (doc_name, label)
grouped = (
    df.groupby(["doc_name", "label"])["text"]
      .apply(lambda x: " ".join(x.astype(str)))
      .reset_index()
)

# Step 2: Build full dictionary
case_dict = {}

for doc_name, sub_df in grouped.groupby("doc_name"):
    # Create label:text mapping for this case
    label_map = dict(zip(sub_df["label"], sub_df["text"]))
    # Assign to master dictionary
    case_dict[doc_name] = label_map

# --- Preview one case ---
first_key = list(case_dict.keys())[0]
print(first_key, "→", case_dict[first_key])


ECLI:NL:GHAMS:2015:2960.txt → {'beoordeling': 'Deze grief slaagt. Het staat vast dat [appellante] een doos met voedingssupplementen heeft achtergelaten bij [geïntimeerde] en dat [geïntimeerde] deze op dat moment heeft behouden en in elk geval een aantal van deze voedingssupplementen heeft gebruikt. De lezing van [appellante] dat de waarde van de doos met voedingssupplementen - naar in hoger beroep niet meer in geschil is: € 2.000,- - zou mogen worden afgetrokken van het geleende bedrag van € 15.000,- vindt bovendien steun in de eerdergenoemde brief van [appellante] , die ook door [geïntimeerde] is ondertekend. In het licht van dit een en ander acht het hof de bestrijding door [geïntimeerde] van de lezing van [appellante] onvoldoende gemotiveerd. Het beroep op verrekening slaagt daarom, zodat de terugbetalingsverplichting van [appellante] nog een bedrag van € 13.000,- beloopt. Van [appellante] had verwacht mogen worden dat zij ook inzicht zou geven in haar vermogenspositie Iedere verder

In [7]:
from bs4 import BeautifulSoup
import os
import pandas as pd

def extract_headers_and_sections_from_xml(xml_path, wanted_titles=None):
    """
    Extracts headers (section titles present in the full text) and their corresponding section texts from a Rechtspraak XML file.
    Returns:
        headers: list of section titles (headers) found in the text
        sections: list of dicts: {"title": title, "text": section_text}
    """
    with open(xml_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'xml')

    sections = []
    for sec in soup.find_all(lambda t: t.name and t.name.lower().endswith('section')):
        title_tag = sec.find(lambda t: t.name and t.name.lower().endswith('title'))
        nr_tag = title_tag.find(lambda t: t.name and t.name.lower().endswith('nr')) if title_tag else None
        nr = nr_tag.text.strip() if nr_tag else ""
        title_text = "".join(title_tag.stripped_strings) if title_tag else ""
        if nr:
            title_text = title_text.replace(nr, "", 1).strip()
        # Get all text in the section (excluding the title/nr)
        section_text = " ".join([s for s in sec.stripped_strings if s not in (nr, title_text)])
        sections.append({"title": title_text, "text": section_text})

    # If wanted_titles is provided, filter headers to only those present in the text
    if wanted_titles:
        wanted_set = set([t.strip().lower() for t in wanted_titles])
        headers = [s["title"] for s in sections if s["title"].strip().lower() in wanted_set]
        filtered_sections = [s for s in sections if s["title"].strip().lower() in wanted_set]
    else:
        headers = [s["title"] for s in sections if s["title"]]
        filtered_sections = [s for s in sections if s["title"]]

    return headers, filtered_sections


# ===========================
# BATCH EXTRACTION (dict form)
# ===========================

xml_folder = 'rechtspraak_xml'

def get_xml_path(ecli):
    ecli = ecli.replace('.txt', '')
    filename = ecli.replace(':', '_') + '.xml'
    return os.path.join(xml_folder, filename)

# Build a list of dicts instead of two separate lists
data = []
df_documents = pd.read_csv('datasets\\documents_raw.csv') 
for ecli in df_documents['doc_name']:
    xml_path = get_xml_path(ecli)
    if os.path.exists(xml_path):
        headers, sections = extract_headers_and_sections_from_xml(xml_path)

        # Convert to dict: { title: text }
        title_text_dict = {s["title"]: s["text"] for s in sections if s.get("title") and s.get("text")}

        data.append({
            "case_name": ecli,
            "extracted_data": title_text_dict
        })
    else:
        data.append({
            "case_name": ecli,
            "extracted_data": None
        })

# Build the final DataFrame
df_final = pd.DataFrame(data)
df_final = df_final[df_final['extracted_data'].apply(lambda d: isinstance(d, dict) and len(d) != 0)].reset_index(drop=True) # Keep only rows with non-empty dicts
df_final


,case_name,extracted_data
0,ECLI:NL:GHAMS:2015:2960.txt,{'Het geding in hoger beroep': 'Partijen worde...
1,ECLI:NL:GHAMS:2015:5282.txt,{'Het verloop van het geding': '1.1 In het ver...
2,ECLI:NL:GHARL:2016:7748.txt,{'Het verdere verloop van het geding in hoger ...
3,ECLI:NL:GHDHA:2016:706.txt,{'GERECHTSHOF DEN HAAG': 'Afdeling Civiel rech...
4,ECLI:NL:GHSHE:2021:1091.txt,"{'In eerste aanleg, hoger beroep en cassatie':..."
...,...,...
95,ECLI:NL:RBNNE:2018:4847.txt,{'Procesverloop': 'Bij besluit van 1 augustus ...
96,ECLI:NL:RBOBR:2019:10.txt,{'Procesverloop': 'Met het besluit van 22 dece...
97,ECLI:NL:RBOVE:2019:1349.txt,{'Procesverloop': 'Bij besluit van 7 mei 2018 ...
98,ECLI:NL:RBROT:2016:4797.txt,{'Procesverloop': 'Bij besluit van 28 november...


In [8]:
## Remove unwanted titles and their sections from the extracted data ##

# Define title groups (section titles to remove)
# Each group contains possible variants/synonyms of a section title
title_groups = {
    "wettelijk_kader": [
        "wettelijk_kader", "toepasselijke wettelijke voorschriften", "de wettelijke voorschriften",
        "toepassende wetsbepalingen", "toepasselijke wetsartikelen", "de toegepaste wettelijke voorschriften",
        "de toepasselijke wetsartikelen", "de toegepaste wettelijke bepalingen"
    ],
    "proceskosten": [
        "proceskosten", "griffierecht en proceskosten", "kosten", "proceskosten en griffierecht"
    ],
    "procesverloop": 
    [
        "procesverloop", "de procedure", "het geding in hoger beroep", "het procesverloop",
        "het geding in eerste aanleg", "ontstaan en loop van het geding", "procesgang",
        "procesverloop in cassatie", "het verloop van de procedure", "procesverloop",
        "het verloop van het geding in eerste aanleg", "het verdere verloop van het geding in hoger beroep",
        "het verloop van het geding", "het verloop van de procedure in hoger beroep",
        "de procedure in hoger beroep", "procesverloop in hoger beroep", "procedure",
        "het verdere verloop van de procedure in hoger beroep", "het verdere verloop van de procedure",
        "de procedure bij de rechtbank", "de procedure in eerste aanleg", "verloop van de procedure",
        "het verdere procesverloop", "procedure bij de rechtbank", "het geschil en de beslissing in eerste aanleg"
    ],
}

import re

def normalize(s):
    """Normalize a title: collapse whitespace, strip and lowercase."""
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

# Flatten to a set of normalized titles to remove
titles_to_remove = set(normalize(t) for vals in title_groups.values() for t in vals)
print(titles_to_remove)

### Remove specified titles from extracted data ###
list_removed = []
list_newd = []
for i,extracted_items in enumerate(df_final.extracted_data,start=1): 
    titles = list(extracted_items.keys())
    n_titles = [normalize(t) for t in titles]
    removed = {}
    for t in n_titles:
        if t in titles_to_remove:
            print(f"Case {i} - Will remove: {t}")
            removed[t] = extracted_items[titles[n_titles.index(t)]]
            extracted_items.pop(titles[n_titles.index(t)])
    list_removed.append(removed)
    list_newd.append(extracted_items if extracted_items else None)
df_final['removed_titles'] = list_removed
df_final

{'de toegepaste wettelijke bepalingen', 'de procedure bij de rechtbank', 'het verloop van de procedure in hoger beroep', 'toepasselijke wettelijke voorschriften', 'procedure', 'toepassende wetsbepalingen', 'het procesverloop', 'ontstaan en loop van het geding', 'de procedure in eerste aanleg', 'het verdere verloop van de procedure', 'het verloop van de procedure', 'procesverloop in cassatie', 'procedure bij de rechtbank', 'het verdere procesverloop', 'procesverloop', 'de procedure', 'toepasselijke wetsartikelen', 'proceskosten', 'procesgang', 'kosten', 'de toepasselijke wetsartikelen', 'het geding in eerste aanleg', 'het verloop van het geding in eerste aanleg', 'griffierecht en proceskosten', 'procesverloop in hoger beroep', 'de toegepaste wettelijke voorschriften', 'de procedure in hoger beroep', 'het geding in hoger beroep', 'het verloop van het geding', 'het verdere verloop van de procedure in hoger beroep', 'het verdere verloop van het geding in hoger beroep', 'de wettelijke voors

,case_name,extracted_data,removed_titles
0,ECLI:NL:GHAMS:2015:2960.txt,{'Feiten': 'De kantonrechter heeft in het best...,{'het geding in hoger beroep': 'Partijen worde...
1,ECLI:NL:GHAMS:2015:5282.txt,{'De gronden van de beslissing': 'Nu van de zi...,{'het verloop van het geding': '1.1 In het ver...
2,ECLI:NL:GHARL:2016:7748.txt,{'De verdere motivering van de beslissing in h...,{'het verdere verloop van het geding in hoger ...
3,ECLI:NL:GHDHA:2016:706.txt,{'GERECHTSHOF DEN HAAG': 'Afdeling Civiel rech...,{}
4,ECLI:NL:GHSHE:2021:1091.txt,"{'In eerste aanleg, hoger beroep en cassatie':...",{}
...,...,...,...
95,ECLI:NL:RBNNE:2018:4847.txt,{'Overwegingen': '1. De rechtbank gaat uit van...,{'procesverloop': 'Bij besluit van 1 augustus ...
96,ECLI:NL:RBOBR:2019:10.txt,{'Overwegingen': 'De aanleiding voor deze proc...,{'procesverloop': 'Met het besluit van 22 dece...
97,ECLI:NL:RBOVE:2019:1349.txt,{'Overwegingen': '1.Op 15 oktober 2017 heeft e...,{'procesverloop': 'Bij besluit van 7 mei 2018 ...
98,ECLI:NL:RBROT:2016:4797.txt,{'Overwegingen': '1. Verweerder heeft aan het ...,{'procesverloop': 'Bij besluit van 28 november...


In [9]:
SEED_WORDS_LIST = {
    "Context": [
        "context_overig", "voorvragen", "de voorvragen", "inleiding", "grondslag en inhoud van het eab",
        "onderzoek van de zaak", "het onderzoek ter terechtzitting", "onderzoek ter terechtzitting", "gronden",
        "het onderzoek op de terechtzitting", "onderzoek op de terechtzitting", "de kern van de zaak",
        "waar gaat deze zaak over?", "inleiding en samenvatting", "waar gaat het over?", "het bewijs",
        "waar gaat de zaak over?"
    ],
    "Feiten": [
        "feiten", "de feiten", "feiten", "de zaak in het kort", "de vaststaande feiten", "vaststaande feiten",
        "uitgangspunten en feiten", "feiten en procesverloop", "feitelijke achtergrond", "uitgangspunten in cassatie"
    ],
    "Beoordeling": [
        "beoordeling_door_rechter", "de beoordeling", "waardering van het bewijs", "de motivering van de beslissing",
        "beoordeling", "bewezenverklaring", "de beoordeling van het bewijs", "de verdere beoordeling",
        "beoordeling van het geschil", "de strafbaarheid van de verdachte", "strafbaarheid van verdachte",
        "de strafbaarheid van verdachte", "de strafbaarheid", "beoordeling door de rechtbank", "strafbaarheid",
        "beoordeling van het middel", "strafbaarheid van de feiten", "de strafbaarheid van het bewezenverklaarde",
        "conclusie en gevolgen", "de bewezenverklaring", "de strafbaarheid van de feiten",
        "de strafbaarheid van het bewezen verklaarde", "beoordeling van het cassatiemiddel",
        "beoordeling van de middelen", "bespreking van het cassatiemiddel", "de bewijsmotivering",
        "beoordeling van het bewijs", "strafbaarheid verdachte", "de motivering van de beslissing in hoger beroep",
        "beoordeling van de ontvankelijkheid van het beroep in cassatie", "de bewijsbeslissing",
        "de kwalificatie van het bewezenverklaarde", "strafbaarheid van de verdachte", "de gronden van de beslissing",
        "motivering van de straf", "de strafbaarheid van het feit", "de beoordeling van het geschil",
        "strafbaarheid van het feit", "motivering straf", "motivering van de sanctie", "het oordeel van de rechtbank",
        "de beoordeling in het incident", "beoordeling van de klachten", "de beoordeling in hoger beroep",
        "overwegingen", "kwalificatie en strafbaarheid van de feiten", "beoordeling van de cassatiemiddelen",
        "beoordeling van het eerste cassatiemiddel", "strafbaarheid feiten",
        "de overwegingen ten aanzien van straf en/of maatregel", "beoordeling van het tweede cassatiemiddel",
        "beoordeling in hoger beroep", "strafbaarheid feit", "motivering van de straffen en maatregelen",
        "kwalificatie en strafbaarheid van het feit", "de beoordeling in conventie en in reconventie",
        "het geschil in reconventie", "de straf en/of de maatregel", "bewijs", "het eerste middel",
        "de beoordeling van de civiele vordering"
    ],
    "Beslissing": [
        "beslissing", "de beslissing", "beslissing", "slotsom", "de uitspraak", "de slotsom", "de strafoplegging",
        "conclusie", "oplegging van straf", "de op te leggen straf of maatregel", "vrijspraak", ".beslissing",
        ". beslissing", "de straf"
    ],
    "Proceshandelingen partijen": [
        "proceshandelingen_partijen", "het geschil", "tenlastelegging", "de tenlastelegging", "de vordering",
        "het verweer", "het verzoek", "geschil", "geding in cassatie", "eis officier van justitie",
        "de vordering en het verweer", "de inhoud van de tenlastelegging", "het wrakingsverzoek",
        "geschil in hoger beroep", "het verzoek en het verweer", "geschil en conclusies van partijen",
        "het cassatieberoep", "het geschil in conventie", "de standpunten", "standpunten van partijen",
        "het standpunt van de officier van justitie", "de vordering tot tenuitvoerlegging"
    ]
}

In [10]:
import re
import os
import pandas as pd
from collections import defaultdict

# Try RapidFuzz; fallback to difflib if not installed
try:
    from rapidfuzz import fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    import difflib
    _HAS_RAPIDFUZZ = False


# -------------------------
# Normalization + tokens
# -------------------------
DUTCH_STOP_TOKENS = {
    "de","het","een","en","van","in","op","aan","bij","voor","met","tot","uit","door",
    "over","onder","naar","als","is","zijn","wordt","werden","was","dat","die","dit",
    "te","ten","ter","der","den","of","maar","ook","nog","dan","wel","niet"
}

def normalize_header(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip(" .:-–—\t")
    return s

def tokenize_for_match(s: str):
    s = normalize_header(s)
    toks = re.findall(r"[a-z0-9]+", s)
    toks = [t for t in toks if t not in DUTCH_STOP_TOKENS]
    return toks


# -------------------------
# REMOVE UNWANTED TITLES
# -------------------------
# Define title groups (section titles to remove)
title_groups = {
    "wettelijk_kader": [
        "wettelijk_kader", "toepasselijke wettelijke voorschriften", "de wettelijke voorschriften",
        "toepassende wetsbepalingen", "toepasselijke wetsartikelen", "de toegepaste wettelijke voorschriften",
        "de toepasselijke wetsartikelen", "de toegepaste wettelijke bepalingen"
    ],
    "proceskosten": [
        "proceskosten", "griffierecht en proceskosten", "kosten", "proceskosten en griffierecht"
    ],
    "procesverloop": [
        "procesverloop", "de procedure", "het geding in hoger beroep", "het procesverloop",
        "het geding in eerste aanleg", "ontstaan en loop van het geding", "procesgang",
        "procesverloop in cassatie", "het verloop van de procedure", "procesverloop",
        "het verloop van het geding in eerste aanleg", "het verdere verloop van het geding in hoger beroep",
        "het verloop van het geding", "het verloop van de procedure in hoger beroep",
        "de procedure in hoger beroep", "procesverloop in hoger beroep", "procedure",
        "het verdere verloop van de procedure in hoger beroep", "het verdere verloop van de procedure",
        "de procedure bij de rechtbank", "de procedure in eerste aanleg", "verloop van de procedure",
        "het verdere procesverloop", "procedure bij de rechtbank", "het geschil en de beslissing in eerste aanleg"
    ],
}

def build_remove_title_set(title_groups: dict) -> set:
    remove_set = set()
    for _, variants in title_groups.items():
        if not isinstance(variants, (list, tuple)):
            continue
        for v in variants:
            if isinstance(v, str):
                nv = normalize_header(v)
                if nv:
                    remove_set.add(nv)
    return remove_set

REMOVE_TITLE_SET = build_remove_title_set(title_groups)

def is_unwanted_title(title: str) -> bool:
    return normalize_header(title) in REMOVE_TITLE_SET


# -------------------------
# Seed structures
# -------------------------
def build_seed_structures(seed_dict):
    """
    seed_dict: {group_name: [phrases...]}

    Returns:
      exact_map: norm_phrase -> group_name
      group_phrase_tokens: group_name -> list of (norm_phrase, token_set)
      all_seed_phrases: list of (group_name, norm_phrase)
      seed_phrase_tokens_flat: list of (group_name, norm_phrase, token_set)
    """
    exact_map = {}
    group_phrase_tokens = defaultdict(list)
    all_seed_phrases = []
    seed_phrase_tokens_flat = []

    for group_name, phrases in seed_dict.items():
        if not isinstance(phrases, (list, tuple)):
            continue

        for phrase in phrases:
            if not isinstance(phrase, str):
                continue

            norm_phrase = normalize_header(phrase)
            if not norm_phrase:
                continue

            # exact map (first occurrence wins)
            if norm_phrase not in exact_map:
                exact_map[norm_phrase] = group_name

            tok_set = set(tokenize_for_match(phrase))
            if tok_set:
                group_phrase_tokens[group_name].append((norm_phrase, tok_set))
                seed_phrase_tokens_flat.append((group_name, norm_phrase, tok_set))

            all_seed_phrases.append((group_name, norm_phrase))

    return exact_map, group_phrase_tokens, all_seed_phrases, seed_phrase_tokens_flat


EXACT_SEED_MAP, GROUP_PHRASE_TOKENS, ALL_SEED_PHRASES, SEED_PHRASE_TOKENS_FLAT = build_seed_structures(SEED_WORDS_LIST)


# -------------------------
# Matching: exact → token overlap → fuzzy
# -------------------------
def token_overlap_match(title: str, min_overlap_tokens: int = 2, min_jaccard: float = 0.34):
    title_tokens = set(tokenize_for_match(title))
    if not title_tokens:
        return (False, None, {})

    best = None
    for group_name, phrase_list in GROUP_PHRASE_TOKENS.items():
        for seed_phrase, seed_tokens in phrase_list:
            inter = title_tokens & seed_tokens
            overlap = len(inter)
            if overlap == 0:
                continue

            union = len(title_tokens | seed_tokens)
            jacc = overlap / union if union else 0.0

            cand = (overlap, jacc, group_name, seed_phrase, inter)
            if best is None or (cand[0], cand[1]) > (best[0], best[1]):
                best = cand

    if best is None:
        return (False, None, {})

    overlap, jacc, group_name, seed_phrase, inter = best
    if overlap >= min_overlap_tokens and jacc >= min_jaccard:
        return (True, group_name, {
            "best_seed": seed_phrase,
            "overlap": overlap,
            "jaccard": jacc,
            "shared_tokens": sorted(list(inter))
        })

    return (False, None, {
        "best_seed": seed_phrase,
        "overlap": overlap,
        "jaccard": jacc,
        "shared_tokens": sorted(list(inter))
    })


def _fuzzy_score(a: str, b: str) -> float:
    if _HAS_RAPIDFUZZ:
        return float(fuzz.token_set_ratio(a, b))
    else:
        return float(difflib.SequenceMatcher(None, a, b).ratio() * 100.0)


def fuzzy_match(title: str,
                min_score: float = 90.0,
                min_title_len: int = 8,
                require_token_overlap: bool = True,
                min_overlap_for_fuzzy: int = 1):
    norm_title = normalize_header(title)
    if len(norm_title) < min_title_len:
        return (False, None, {"reason": "title_too_short", "norm_title": norm_title})

    title_tokens = set(tokenize_for_match(title))

    best = None
    for group_name, seed_phrase in ALL_SEED_PHRASES:
        score = _fuzzy_score(norm_title, seed_phrase)
        if best is None or score > best[0]:
            best = (score, group_name, seed_phrase)

    if best is None:
        return (False, None, {"reason": "no_seeds", "norm_title": norm_title})

    score, group_name, seed_phrase = best

    debug = {"norm_title": norm_title, "best_seed": seed_phrase, "score": score}

    if score < min_score:
        debug["reason"] = "score_below_threshold"
        return (False, None, debug)

    if require_token_overlap:
        seed_tokens = set(tokenize_for_match(seed_phrase))
        inter = title_tokens & seed_tokens
        debug["shared_tokens"] = sorted(list(inter))
        debug["overlap"] = len(inter)
        if len(inter) < min_overlap_for_fuzzy:
            debug["reason"] = "no_token_overlap_with_best_seed"
            return (False, None, debug)

    return (True, group_name, debug)


def match_group_cascade(title: str,
                        token_min_overlap: int = 2,
                        token_min_jaccard: float = 0.34,
                        fuzzy_min_score: float = 90.0,
                        fuzzy_min_title_len: int = 8,
                        fuzzy_require_token_overlap: bool = True,
                        fuzzy_min_overlap: int = 1):
    norm_title = normalize_header(title)
    if not norm_title:
        return (False, None, norm_title, None, {})

    # 1) Exact
    group = EXACT_SEED_MAP.get(norm_title)
    if group is not None:
        return (True, group, norm_title, "exact", {"norm_title": norm_title})

    # 2) Token overlap
    ok2, group2, dbg2 = token_overlap_match(title,
                                            min_overlap_tokens=token_min_overlap,
                                            min_jaccard=token_min_jaccard)
    if ok2:
        dbg2["norm_title"] = norm_title
        return (True, group2, norm_title, "token_overlap", dbg2)

    # 3) Fuzzy
    ok3, group3, dbg3 = fuzzy_match(title,
                                    min_score=fuzzy_min_score,
                                    min_title_len=fuzzy_min_title_len,
                                    require_token_overlap=fuzzy_require_token_overlap,
                                    min_overlap_for_fuzzy=fuzzy_min_overlap)
    if ok3:
        return (True, group3, norm_title, "fuzzy", dbg3)

    return (False, None, norm_title, None, {"norm_title": norm_title})


# -------------------------
# Main extraction loop + audits + unmatched + removed
# -------------------------
filtered_rows = []

token_overlap_audit_rows = []
fuzzy_audit_rows = []

unmatched_by_case = defaultdict(list)
unmatched_rows = []

# optional: track removed titles
removed_rows = []

for ecli in df_documents["doc_name"]:
    xml_path = get_xml_path(ecli)
    if not os.path.exists(xml_path):
        continue

    headers, sections = extract_headers_and_sections_from_xml(xml_path, wanted_titles=None)

    kept = []
    for s in sections:
        title = s.get("title", "")
        text = s.get("text", "")
        norm_title = normalize_header(title)

        # ---- REMOVE unwanted sections BEFORE matching ----
        if title and norm_title and is_unwanted_title(title):
            removed_rows.append({
                "case_name": ecli,
                "title": title,
                "norm_title": norm_title,
                "section_text_snippet": (text or "")[:200]
            })
            continue

        ok, group_name, norm_title, method, debug = match_group_cascade(
            title,
            token_min_overlap=2,
            token_min_jaccard=0.34,
            fuzzy_min_score=80.0,
            fuzzy_min_title_len=8,
            fuzzy_require_token_overlap=True,
            fuzzy_min_overlap=1
        )

        if ok:
            kept.append({
                "title": title,
                "text": text,
                "hdr_group": group_name,
                "norm_title": norm_title,
                "match_method": method,
                "match_debug": debug
            })

            if method == "token_overlap":
                token_overlap_audit_rows.append({
                    "case_name": ecli,
                    "original_title": title,
                    "normalized_title": norm_title,
                    "assigned_group": group_name,
                    "best_seed": debug.get("best_seed"),
                    "shared_tokens": debug.get("shared_tokens"),
                    "overlap": debug.get("overlap"),
                    "jaccard": debug.get("jaccard"),
                    "section_text_snippet": (text or "")[:300]
                })

            elif method == "fuzzy":
                fuzzy_audit_rows.append({
                    "case_name": ecli,
                    "original_title": title,
                    "normalized_title": norm_title,
                    "assigned_group": group_name,
                    "best_seed": debug.get("best_seed"),
                    "fuzzy_score": debug.get("score"),
                    "shared_tokens": debug.get("shared_tokens"),
                    "overlap": debug.get("overlap"),
                    "section_text_snippet": (text or "")[:300],
                    "used_rapidfuzz": _HAS_RAPIDFUZZ
                })

        else:
            if title and norm_title:
                entry = {
                    "title": title,
                    "norm_title": norm_title,
                    "section_text_snippet": (text or "")[:300],
                }
                unmatched_by_case[ecli].append(entry)
                unmatched_rows.append({"case_name": ecli, **entry})

    if kept:
        title_text_dict = {k["title"]: k["text"] for k in kept if k["title"] and k["text"]}
        filtered_rows.append({
            "case_name": ecli,
            "extracted_data": title_text_dict,
            "matched_headers": [k["title"] for k in kept],
            "matched_groups": [k["hdr_group"] for k in kept],
            "matched_norm_titles": [k["norm_title"] for k in kept],
            "match_methods": [k["match_method"] for k in kept],
        })

df_filtered_cascade = pd.DataFrame(filtered_rows).reset_index(drop=True)
df_token_overlap_audit = pd.DataFrame(token_overlap_audit_rows)
df_fuzzy_audit = pd.DataFrame(fuzzy_audit_rows)
df_unmatched = pd.DataFrame(unmatched_rows).reset_index(drop=True)
df_removed = pd.DataFrame(removed_rows).reset_index(drop=True)

print("Cases with ≥1 unmatched header:", len(unmatched_by_case))
print("Total unmatched headers:", len(df_unmatched))
print("Total removed sections:", len(df_removed))

df_filtered_cascade, df_token_overlap_audit, df_fuzzy_audit, df_unmatched, df_removed

df_filtered_cascade['number_of_headers'] = df_filtered_cascade.matched_headers.apply(lambda d: len(d))
df_filtered_cascade


Cases with ≥1 unmatched header: 62
Total unmatched headers: 128
Total removed sections: 86


,case_name,extracted_data,matched_headers,matched_groups,matched_norm_titles,match_methods,number_of_headers
0,ECLI:NL:GHAMS:2015:2960.txt,{'Feiten': 'De kantonrechter heeft in het best...,"[Feiten, Beoordeling, Beslissing]","[Feiten, Beoordeling, Beslissing]","[feiten, beoordeling, beslissing]","[exact, exact, exact]",3
1,ECLI:NL:GHAMS:2015:5282.txt,{'De gronden van de beslissing': 'Nu van de zi...,"[De gronden van de beslissing, De beslissing]","[Beoordeling, Beslissing]","[de gronden van de beslissing, de beslissing]","[exact, exact]",2
2,ECLI:NL:GHARL:2016:7748.txt,{'De verdere motivering van de beslissing in h...,[De verdere motivering van de beslissing in ho...,"[Beoordeling, Beslissing]",[de verdere motivering van de beslissing in ho...,"[token_overlap, exact]",2
3,ECLI:NL:GHDHA:2016:706.txt,{'Inhoudelijke beoordeling': '45. Op 1 januari...,"[Inhoudelijke beoordeling, Inhoudelijke beoord...","[Beoordeling, Beoordeling, Beoordeling, Beoord...","[inhoudelijke beoordeling, inhoudelijke beoord...","[fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy]",7
4,ECLI:NL:GHSHE:2021:1091.txt,{'De beoordeling': '2.1. De Hoge Raad heeft in...,"[De beoordeling, De uitspraak]","[Beoordeling, Beslissing]","[de beoordeling, de uitspraak]","[exact, exact]",2
...,...,...,...,...,...,...,...
95,ECLI:NL:RBNNE:2018:4847.txt,{'Overwegingen': '1. De rechtbank gaat uit van...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2
96,ECLI:NL:RBOBR:2019:10.txt,{'Overwegingen': 'De aanleiding voor deze proc...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2
97,ECLI:NL:RBOVE:2019:1349.txt,{'Overwegingen': '1.Op 15 oktober 2017 heeft e...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2
98,ECLI:NL:RBROT:2016:4797.txt,{'Overwegingen': '1. Verweerder heeft aan het ...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2


In [11]:
# Unique headers (original titles)
unique_headers = (
    df_filtered_cascade["matched_norm_titles"]
    .explode()
    .dropna()
    .sort_values()
    .unique()
)

print(f"Number of unique headers: {len(unique_headers)}")
unique_headers


Number of unique headers: 143


array(['6. beslissing',
       'aan de beoordeling van het middel voorafgaande opmerkingen',
       'beoordeling', 'beoordeling van de middelen',
       'beoordeling van de vordering', 'beoordeling van het bewijs',
       'beoordeling van het geschil', 'beoordeling van het hoger beroep',
       'beoordeling van het middel',
       'beoordeling van het middel in het incidentele beroep',
       'beoordeling van het middel in het principale beroep',
       'beschikking op het verzoek van', 'beslissing',
       'beslissing op de vordering van de benadeelde partij [slachtoffer 2]',
       'beslissing op de vordering van de benadeelde partij [slachtoffer 3]',
       'beslissing op het hoger beroep', 'bewezenverklaring',
       'bewezenverklaring en bewijsvoering', 'bewijs',
       'bewijsoverweging ten aanzien van de feiten 1 en 2',
       'de beoordeling', 'de beoordeling in conventie',
       'de beoordeling in conventie en in reconventie',
       'de beoordeling in reconventie', 'de beoor

In [12]:
unique_header_groups = (
    df_filtered_cascade["matched_groups"]
    .explode()
    .dropna()
    .sort_values()
    .unique()
)

print(f"Number of unique header groups: {len(unique_header_groups)}")
unique_header_groups


Number of unique header groups: 5


array(['Beoordeling', 'Beslissing', 'Context', 'Feiten',
       'Proceshandelingen partijen'], dtype=object)

In [13]:
header_group_map = (
    df_filtered_cascade
    .explode(["matched_headers", "matched_groups"])
    .loc[:, ["matched_headers", "matched_groups"]]
    .drop_duplicates()
    .sort_values(["matched_groups", "matched_headers"])
    .reset_index(drop=True)
)

header_group_map

header_group_dict = (
    df_filtered_cascade
    .explode(["matched_headers", "matched_groups"])
    .loc[:, ["matched_headers", "matched_groups"]]
    .drop_duplicates()
    .groupby("matched_groups")["matched_headers"]
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
)

header_group_dict



{'Beoordeling': ['Aan de beoordeling van het middel voorafgaande opmerkingen',
  'BEOORDELING VAN DE VORDERING',
  'BEOORDELING VAN HET HOGER BEROEP',
  'BESLISSING OP HET HOGER BEROEP',
  'Beoordeling',
  'Beoordeling van de middelen',
  'Beoordeling van het bewijs',
  'Beoordeling van het geschil',
  'Beoordeling van het hoger beroep',
  'Beoordeling van het middel',
  'Beoordeling van het middel in het incidentele beroep',
  'Beoordeling van het middel in het principale beroep',
  'Bewezenverklaring',
  'Bewezenverklaring en bewijsvoering',
  'Bewijs',
  'De beoordeling',
  'De beoordeling in conventie',
  'De beoordeling in conventie en in reconventie',
  'De beoordeling in reconventie',
  'De beoordeling van het bewijs',
  'De bewezenverklaring',
  'De bewezenverklaring.',
  'De gronden van de beslissing',
  'De kwalificatie van het bewezenverklaarde',
  'De ontvankelijkheid van het beroep',
  'De straf en/of de maatregel',
  'De strafbaarheid',
  'De strafbaarheid van de feiten',

In [14]:
df_final = df_filtered_cascade.copy()
df_final['data']=df_final['extracted_data'].apply(lambda d: ''.join(list(d.values())) if isinstance(d, dict) else None)
df_final

,case_name,extracted_data,matched_headers,matched_groups,matched_norm_titles,match_methods,number_of_headers,data
0,ECLI:NL:GHAMS:2015:2960.txt,{'Feiten': 'De kantonrechter heeft in het best...,"[Feiten, Beoordeling, Beslissing]","[Feiten, Beoordeling, Beslissing]","[feiten, beoordeling, beslissing]","[exact, exact, exact]",3,De kantonrechter heeft in het bestreden vonnis...
1,ECLI:NL:GHAMS:2015:5282.txt,{'De gronden van de beslissing': 'Nu van de zi...,"[De gronden van de beslissing, De beslissing]","[Beoordeling, Beslissing]","[de gronden van de beslissing, de beslissing]","[exact, exact]",2,Nu van de zijde van de partijen geen bezwaren ...
2,ECLI:NL:GHARL:2016:7748.txt,{'De verdere motivering van de beslissing in h...,[De verdere motivering van de beslissing in ho...,"[Beoordeling, Beslissing]",[de verdere motivering van de beslissing in ho...,"[token_overlap, exact]",2,2.1 Bij voormeld tussenarrest heeft het hof pa...
3,ECLI:NL:GHDHA:2016:706.txt,{'Inhoudelijke beoordeling': '45. Op 1 januari...,"[Inhoudelijke beoordeling, Inhoudelijke beoord...","[Beoordeling, Beoordeling, Beoordeling, Beoord...","[inhoudelijke beoordeling, inhoudelijke beoord...","[fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy]",7,45. Op 1 januari 2002 is een nieuw Turks Burge...
4,ECLI:NL:GHSHE:2021:1091.txt,{'De beoordeling': '2.1. De Hoge Raad heeft in...,"[De beoordeling, De uitspraak]","[Beoordeling, Beslissing]","[de beoordeling, de uitspraak]","[exact, exact]",2,2.1. De Hoge Raad heeft in zijn voornoemde arr...
...,...,...,...,...,...,...,...,...
95,ECLI:NL:RBNNE:2018:4847.txt,{'Overwegingen': '1. De rechtbank gaat uit van...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1. De rechtbank gaat uit van de volgende relev...
96,ECLI:NL:RBOBR:2019:10.txt,{'Overwegingen': 'De aanleiding voor deze proc...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,De aanleiding voor deze procedure 1. Eiser wil...
97,ECLI:NL:RBOVE:2019:1349.txt,{'Overwegingen': '1.Op 15 oktober 2017 heeft e...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1.Op 15 oktober 2017 heeft eiser verweerder ve...
98,ECLI:NL:RBROT:2016:4797.txt,{'Overwegingen': '1. Verweerder heeft aan het ...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1. Verweerder heeft aan het bestreden besluit ...


In [15]:
df_final.data[0]

'De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen. Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.3.1. Op 10 oktober 2012 heeft [geïntimeerde] een bedrag van € 15.000,- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoop van voedingssupplementen) te beginnen. Op 4/5 januari 2013 heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben ondertekend. Deze verklaring houdt onder meer in: “Dit bedrag wordt in gedeelten terugbetaald; wanneer er door deze promotie, inschrijvingen en verdiensten in welke vorm dan ook; ontvangen worden; gaat daar zo snel mogelijk het grootste deel direct van terug/retour naar de rekening van dhr. [geïntimeerde] (…) Zoals afgesproken; zo ook: zou er een grote doos met voedingssupplementen ter waarde van € 2.000,= bij dhr. v. Diest terecht komen v

In [16]:
import re

def replace_money(text):
    if pd.isna(text):
        return text
    text = str(text)
    # covers € / eur / euro followed by numbers, commas, dots, or spaces
    text = re.sub(r"(€|eur|euro)\s?[\d\.\,]+", "[geld]", text, flags=re.IGNORECASE)
    return text

def replace_dates_by_month_window(text):
    if pd.isna(text):
        return text
    text = str(text)
    months = [
        "januari","februari","maart","april","mei","juni","juli",
        "augustus","september","oktober","november","december",
        "jan","feb","mrt","apr","jun","jul","aug","sep","okt","nov","dec"
    ]
    pattern = r"(?:\b\S+\s)?\b(" + "|".join(months) + r")\b(?:\s\S+)?"
    text = re.sub(pattern, "[datum]", text, flags=re.IGNORECASE)
    return text

import re
import pandas as pd

# Compile once for speed
PARTICLES = r"(?:van|de|den|der|het|ten|ter|te|la|le|l'|d')"
SURNAME = r"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ\-']+"
INITIALS_DOTTED = r"(?:[A-Z]\.\s*){1,5}"
HONORIFICS = r"(?:mr|mrs|mevr|mw|meneer|dr|drs|prof|prof\.dr)\.?"

# 1) Honorific + initials + surname  (e.g., "mr. P. Oudkerk", "MR A.B. de Vries")
pat_honorific_initials = re.compile(
    rf"\b{HONORIFICS}\s+{INITIALS_DOTTED}(?:{PARTICLES}\s+)?{SURNAME}\b",
    flags=re.IGNORECASE
)

# 2) Initials + surname (no honorific)  (e.g., "J.E. Molenaar", "C.M. Aarts")
pat_initials_surname = re.compile(
    rf"\b{INITIALS_DOTTED}(?:{PARTICLES}\s+)?{SURNAME}\b"
)

# 3) Optional broader form: Firstname + surname (riskier — leave OFF by default)
#    e.g., "Jan de Vries" | enable by setting BROAD_FIRSTNAME=True
BROAD_FIRSTNAME = False
pat_firstname_surname = re.compile(
    rf"\b[A-Z][a-zà-öø-ÿ]{{2,}}\s+(?:{PARTICLES}\s+)?{SURNAME}\b"
)

def replace_person_names(text, broad_firstname=BROAD_FIRSTNAME):
    if pd.isna(text): 
        return text
    s = str(text)

    # Protect placeholders like [appellante], [geïntimeerde], [datum], [geld]
    protected = {}
    def _protect(m):
        key = f"§§§{len(protected)}§§§"
        protected[key] = m.group(0)
        return key
    s = re.sub(r"\[[^\]\n]{1,50}\]", _protect, s)

    s = pat_honorific_initials.sub("[naam]", s)
    s = pat_initials_surname.sub("[naam]", s)
    if broad_firstname:
        s = pat_firstname_surname.sub("[naam]", s)

    # Restore placeholders
    for k, v in protected.items():
        s = s.replace(k, v)
    return s

# EXAMPLE: apply to your dataframe
# df = pd.read_csv("your_dataset.csv")

# print(df[["data_clean"]].head(10))





df_final["data_clean"] = df_final["data"].apply(replace_money)
df_final["data_clean"] = df_final["data_clean"].apply(replace_dates_by_month_window)
df_final["data_clean"] = df_final["data_clean"].astype(str).map(replace_person_names)

#df_final["data_clean"] = df_final["data_clean"].apply(replace_dates)
df_final.data_clean[0]


'De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen. Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.3.1. Op [datum] heeft [geïntimeerde] een bedrag van [geld]- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoop van voedingssupplementen) te beginnen. Op [datum] heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben ondertekend. Deze verklaring houdt onder meer in: “Dit bedrag wordt in gedeelten terugbetaald; wanneer er door deze promotie, inschrijvingen en verdiensten in welke vorm dan ook; ontvangen worden; gaat daar zo snel mogelijk het grootste deel direct van terug/retour naar de rekening van dhr. [geïntimeerde] (…) Zoals afgesproken; zo ook: zou er een grote doos met voedingssupplementen ter waarde van [geld]= bij dhr. v. Diest terecht komen voor eigen gebruik (…) 

## Sentence Splitter

In [17]:
### Split the text data into sentences ###
from sentence_splitter import sentence_splitter
ss = sentence_splitter()


df = df_final[['case_name','data_clean']].rename(columns={"data_clean":"text"})
df['label'] = ["None" for _ in range(len(df))]
text_col = "text"  #
# keep everything else as metadata
meta_cols = ["case_name"]
xml_data = ss.dataframe_to_sentences(df, text_col="text", label_col="label", meta_cols=meta_cols)
#xml_data.to_csv(r"final/annotations_sentences_clean_v4.csv", index=False)
xml_data.head()



,block_row,sent_index,sent_text,sent_len,label,case_name
0,1,1,De kantonrechter heeft in het bestreden vonnis...,136,None,ECLI:NL:GHAMS:2015:2960.txt
1,1,2,Deze feiten zijn in hoger beroep niet in gesch...,97,None,ECLI:NL:GHAMS:2015:2960.txt
2,1,3,Op [datum] heeft [geïntimeerde] een bedrag van...,200,None,ECLI:NL:GHAMS:2015:2960.txt
3,1,4,Op [datum] heeft [appellante] een schriftelijk...,106,None,ECLI:NL:GHAMS:2015:2960.txt
4,1,5,Deze verklaring houdt onder meer in: “Dit bedr...,278,None,ECLI:NL:GHAMS:2015:2960.txt


In [18]:
### Print sentences with labels ###
for ele, label in zip(xml_data.sent_text,xml_data.label):
    print(ele + "====>[" + str(label) + "]")
    print("#####")

De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen.====>[None]
#####
Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.====>[None]
#####
Op [datum] heeft [geïntimeerde] een bedrag van [geld]- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoop van voedingssupplementen) te beginnen.====>[None]
#####
Op [datum] heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben onderteken====>[None]
#####
Deze verklaring houdt onder meer in: “Dit bedrag wordt in gedeelten terugbetaald; wanneer er door deze promotie, inschrijvingen en verdiensten in welke vorm dan ook; ontvangen worden; gaat daar zo snel mogelijk het grootste deel direct van terug/retour naar de rekening van dhr.====>[None]
#####
[geïntimeerde] (...) Zoals afgesproken; zo ook: zou er een grote doos met voedingssupplemente

In [19]:
### Match sentences to their labels based on the extracted sections ###
# Some sentences may not find a matching label and will be assigned "None"

CASE = []
SENTENCES = []
LABELS = []
for case, text in zip(xml_data.case_name,xml_data.sent_text):
    #print(case + "====>[" + text + "]")
    found  = False
    for labels in case_dict[case]: 
        text_to_check = case_dict[case][labels].lower()
        text_to_check = re.sub(r'\\n+', ' ', text_to_check)
        text_to_check = re.sub(r'\s+', ' ', text_to_check).strip()
        text_to_check = replace_dates_by_month_window(text_to_check)
        text_to_check = replace_money(text_to_check)
        text_to_check = replace_person_names(text_to_check)
        if text.lower() in text_to_check:
            #print("Found label:", labels)
            CASE.append(case)
            SENTENCES.append(text)
            LABELS.append(labels)
            found = True
            break
    if not found:
        #print("Not found:", text)
        CASE.append(case)
        SENTENCES.append(text)
        LABELS.append("None")

model_data = {
    "case_name": CASE,  
    "sent_text": SENTENCES,
    "label": LABELS
}
model_data = pd.DataFrame(model_data)

In [20]:
model_data

,case_name,sent_text,label
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None
...,...,...,...
8456,ECLI:NL:RBZWB:2015:638.txt,Het bestreden besluit kan in stand blijven.,beoordeling
8457,ECLI:NL:RBZWB:2015:638.txt,Er is geen grond voor een proceskostenveroorde...,None
8458,ECLI:NL:RBZWB:2015:638.txt,De rechtbank verklaart het beroep ongegron,beslissing
8459,ECLI:NL:RBZWB:2015:638.txt,"Deze uitspraak is gedaan door [naam], voorzitt...",None


In [21]:
df_final

,case_name,extracted_data,matched_headers,matched_groups,matched_norm_titles,match_methods,number_of_headers,data,data_clean
0,ECLI:NL:GHAMS:2015:2960.txt,{'Feiten': 'De kantonrechter heeft in het best...,"[Feiten, Beoordeling, Beslissing]","[Feiten, Beoordeling, Beslissing]","[feiten, beoordeling, beslissing]","[exact, exact, exact]",3,De kantonrechter heeft in het bestreden vonnis...,De kantonrechter heeft in het bestreden vonnis...
1,ECLI:NL:GHAMS:2015:5282.txt,{'De gronden van de beslissing': 'Nu van de zi...,"[De gronden van de beslissing, De beslissing]","[Beoordeling, Beslissing]","[de gronden van de beslissing, de beslissing]","[exact, exact]",2,Nu van de zijde van de partijen geen bezwaren ...,Nu van de zijde van de partijen geen bezwaren ...
2,ECLI:NL:GHARL:2016:7748.txt,{'De verdere motivering van de beslissing in h...,[De verdere motivering van de beslissing in ho...,"[Beoordeling, Beslissing]",[de verdere motivering van de beslissing in ho...,"[token_overlap, exact]",2,2.1 Bij voormeld tussenarrest heeft het hof pa...,2.1 Bij voormeld tussenarrest heeft het hof pa...
3,ECLI:NL:GHDHA:2016:706.txt,{'Inhoudelijke beoordeling': '45. Op 1 januari...,"[Inhoudelijke beoordeling, Inhoudelijke beoord...","[Beoordeling, Beoordeling, Beoordeling, Beoord...","[inhoudelijke beoordeling, inhoudelijke beoord...","[fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy, fuzzy]",7,45. Op 1 januari 2002 is een nieuw Turks Burge...,45. Op [datum] is een nieuw Turks Burgerlijk W...
4,ECLI:NL:GHSHE:2021:1091.txt,{'De beoordeling': '2.1. De Hoge Raad heeft in...,"[De beoordeling, De uitspraak]","[Beoordeling, Beslissing]","[de beoordeling, de uitspraak]","[exact, exact]",2,2.1. De Hoge Raad heeft in zijn voornoemde arr...,2.1. De Hoge Raad heeft in zijn voornoemde arr...
...,...,...,...,...,...,...,...,...,...
95,ECLI:NL:RBNNE:2018:4847.txt,{'Overwegingen': '1. De rechtbank gaat uit van...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1. De rechtbank gaat uit van de volgende relev...,1. De rechtbank gaat uit van de volgende relev...
96,ECLI:NL:RBOBR:2019:10.txt,{'Overwegingen': 'De aanleiding voor deze proc...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,De aanleiding voor deze procedure 1. Eiser wil...,De aanleiding voor deze procedure 1. Eiser wil...
97,ECLI:NL:RBOVE:2019:1349.txt,{'Overwegingen': '1.Op 15 oktober 2017 heeft e...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1.Op 15 oktober 2017 heeft eiser verweerder ve...,1.Op [datum] heeft eiser verweerder verzocht o...
98,ECLI:NL:RBROT:2016:4797.txt,{'Overwegingen': '1. Verweerder heeft aan het ...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",2,1. Verweerder heeft aan het bestreden besluit ...,1. Verweerder heeft aan het bestreden besluit ...


In [22]:
import re
from sentence_splitter import sentence_splitter

spl = sentence_splitter()

def canonical_clean(text: str) -> str:
    """
    One cleaning function used for:
    - section blocks
    - individual sentences
    So substring matching works reliably.
    """
    if text is None:
        return ""

    # 1) your splitter cleaner (keeps paragraphs logic consistent with your pipeline)
    t = spl.clean_block_preserve_paragraphs(str(text))

    # 2) your additional cleaning (exactly as you asked)
    t = re.sub(r'\\n+', ' ', t)          # literal \n (if present)
    t = re.sub(r'\n+', ' ', t)           # real newlines
    t = re.sub(r'\s+', ' ', t).strip()

    t = replace_dates_by_month_window(t)
    t = replace_money(t)
    t = replace_person_names(t)

    # final whitespace normalize
    t = re.sub(r'\s+', ' ', t).strip()
    return t



In [23]:
df_final.extracted_data[0]

{'Feiten': 'De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen. Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.',
 'Beoordeling': '3.1. Op 10 oktober 2012 heeft [geïntimeerde] een bedrag van € 15.000,- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoop van voedingssupplementen) te beginnen. Op 4/5 januari 2013 heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben ondertekend. Deze verklaring houdt onder meer in: “Dit bedrag wordt in gedeelten terugbetaald; wanneer er door deze promotie, inschrijvingen en verdiensten in welke vorm dan ook; ontvangen worden; gaat daar zo snel mogelijk het grootste deel direct van terug/retour naar de rekening van dhr. [geïntimeerde] (…) Zoals afgesproken; zo ook: zou er een grote doos met voedingssupplementen ter waarde van € 2.000,= bi

In [24]:
import pandas as pd

SECTIONS_DF_NAME = "df_final"   # <- replace with your actual df name
sections_df = globals()[SECTIONS_DF_NAME]

assert "case_name" in sections_df.columns, "sections_df must have case_name"
assert "extracted_data" in sections_df.columns, "sections_df must have extracted_data"
assert "case_name" in model_data.columns, "model_data must have case_name"
assert "sent_text" in model_data.columns, "model_data must have sent_text"

print("sections_df shape:", sections_df.shape)
print("model_data shape:", model_data.shape)

# Build lookup: case_name -> extracted_data dict
case2sections = dict(zip(sections_df["case_name"], sections_df["extracted_data"]))

# Output columns
model_data["hdr_title"] = None
model_data["hdr_match"] = 0

total = len(model_data)
matched = 0

matched_examples = []
unmatched_examples = []

for case_name, case_df in model_data.groupby("case_name", sort=False):
    sec_dict = case2sections.get(case_name)

    if not isinstance(sec_dict, dict) or len(sec_dict) == 0:
        continue

    # Clean ALL blocks once
    headers = []
    blocks_clean = []
    for hdr, block in sec_dict.items():
        if block is None or not str(block).strip():
            continue
        headers.append(hdr)
        blocks_clean.append(canonical_clean(block))

    if not headers:
        continue

    # Assign each sentence to FIRST matching header block
    for idx, sent in zip(case_df.index.tolist(), case_df["sent_text"].tolist()):
        s_clean = canonical_clean(sent)
        if not s_clean:
            continue

        found_hdr = None
        for hdr, b_clean in zip(headers, blocks_clean):
            if s_clean in b_clean:
                found_hdr = hdr
                break

        if found_hdr is not None:
            model_data.at[idx, "hdr_title"] = found_hdr
            model_data.at[idx, "hdr_match"] = 1
            matched += 1
            if len(matched_examples) < 12:
                matched_examples.append((case_name, found_hdr, sent[:160]))
        else:
            if len(unmatched_examples) < 12:
                unmatched_examples.append((case_name, sent[:160]))

print("\n=== HEADER MATCH RESULTS ===")
print("Total sentences:", total)
print("Matched sentences:", matched)
print("Coverage:", round(matched / max(1, total), 3))

print("\n--- Matched examples (up to 12) ---")
for c, h, s in matched_examples:
    print(f"CASE={c} | HDR={h} | SENT={s}")

print("\n--- Unmatched examples (up to 12) ---")
for c, s in unmatched_examples:
    print(f"CASE={c} | SENT={s}")

print("\nTop hdr_title values (including NaN):")
print(model_data["hdr_title"].value_counts(dropna=False).head(20))


sections_df shape: (100, 9)
model_data shape: (8461, 3)

=== HEADER MATCH RESULTS ===
Total sentences: 8461
Matched sentences: 8391
Coverage: 0.992

--- Matched examples (up to 12) ---
CASE=ECLI:NL:GHAMS:2015:2960.txt | HDR=Feiten | SENT=De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen.
CASE=ECLI:NL:GHAMS:2015:2960.txt | HDR=Feiten | SENT=Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.
CASE=ECLI:NL:GHAMS:2015:2960.txt | HDR=Beoordeling | SENT=Op [datum] heeft [geïntimeerde] een bedrag van [geld]- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoo
CASE=ECLI:NL:GHAMS:2015:2960.txt | HDR=Beoordeling | SENT=Op [datum] heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben onderteken
CASE=ECLI:NL:GHAMS:2015:2960.txt | HDR=Beoordeling | SENT=Deze verklaring houdt onder meer

In [25]:
model_data

,case_name,sent_text,label,hdr_title,hdr_match
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1
...,...,...,...,...,...
8456,ECLI:NL:RBZWB:2015:638.txt,Het bestreden besluit kan in stand blijven.,beoordeling,Overwegingen,1
8457,ECLI:NL:RBZWB:2015:638.txt,Er is geen grond voor een proceskostenveroorde...,None,Overwegingen,1
8458,ECLI:NL:RBZWB:2015:638.txt,De rechtbank verklaart het beroep ongegron,beslissing,Beslissing,1
8459,ECLI:NL:RBZWB:2015:638.txt,"Deze uitspraak is gedaan door [naam], voorzitt...",None,Beslissing,1


In [26]:
# df_sent = your sentence-level dataframe (the one with hdr_title)
# header_group_dict = { "Beoordeling": [...headers...], "Feiten": [...], ... }

# 1) Invert mapping: header -> group
header_to_group = {
    header: group
    for group, headers in header_group_dict.items()
    for header in headers
}

# 2) Assign group based on hdr_title; default "Other" if not found
model_data["hdr_group"] = model_data["hdr_title"].map(header_to_group).fillna("Other")

# (optional) quick check
model_data["hdr_group"].value_counts()

hdr_group
Beoordeling                   5122
Feiten                        1208
Beslissing                     964
Proceshandelingen partijen     855
Context                        242
Other                           70
Name: count, dtype: int64

In [27]:
model_data

,case_name,sent_text,label,hdr_title,hdr_match,hdr_group
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling
...,...,...,...,...,...,...
8456,ECLI:NL:RBZWB:2015:638.txt,Het bestreden besluit kan in stand blijven.,beoordeling,Overwegingen,1,Beoordeling
8457,ECLI:NL:RBZWB:2015:638.txt,Er is geen grond voor een proceskostenveroorde...,None,Overwegingen,1,Beoordeling
8458,ECLI:NL:RBZWB:2015:638.txt,De rechtbank verklaart het beroep ongegron,beslissing,Beslissing,1,Beslissing
8459,ECLI:NL:RBZWB:2015:638.txt,"Deze uitspraak is gedaan door [naam], voorzitt...",None,Beslissing,1,Beslissing


In [28]:
model_data.case_name.nunique()

100

In [29]:
model_data = model_data[model_data.hdr_group!='Other']

In [30]:
model_data['label'].value_counts()

label
None                 3060
beoordeling          2293
materiele feiten     1529
proceshandelingen    1263
beslissing            246
Name: count, dtype: int64

In [31]:
model_data.hdr_title.nunique()


143

In [32]:
import json

hdr_group_to_titles = (
    model_data.groupby("hdr_group")["hdr_title"]
      .unique()
      .apply(list)
      .to_dict()
)

with open("hdr_group_to_titles.json", "w", encoding="utf-8") as f:
    json.dump(hdr_group_to_titles, f, ensure_ascii=False, indent=2)

hdr_group_to_titles


{'Beoordeling': ['Beoordeling',
  'De gronden van de beslissing',
  'De verdere motivering van de beslissing in hoger beroep',
  'Inhoudelijke beoordeling',
  'De beoordeling',
  'Beoordeling van het middel in het principale beroep',
  'Beoordeling van het middel in het incidentele beroep',
  'Het verzoek en de beoordeling',
  'De beoordeling in conventie',
  'De beoordeling in reconventie',
  'Kwalificatie van het bewezen verklaarde',
  'Het hoger beroep',
  'Bewezenverklaring',
  'Strafbaarheid van het bewezenverklaarde',
  'Strafbaarheid van de verdachte',
  'Bewezenverklaring en bewijsvoering',
  'Aan de beoordeling van het middel voorafgaande opmerkingen',
  'Waardering van het bewijs',
  'De strafbaarheid van het feit en van verdachte',
  'De strafbaarheid van het bewezenverklaarde en de verdachte',
  'De kwalificatie van het bewezenverklaarde',
  'De strafbaarheid van het feit',
  'De strafbaarheid van de verdachte',
  'Overwegingen ten aanzien van straf en/of maatregel',
  'De 

In [33]:
model_data

,case_name,sent_text,label,hdr_title,hdr_match,hdr_group
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling
...,...,...,...,...,...,...
8456,ECLI:NL:RBZWB:2015:638.txt,Het bestreden besluit kan in stand blijven.,beoordeling,Overwegingen,1,Beoordeling
8457,ECLI:NL:RBZWB:2015:638.txt,Er is geen grond voor een proceskostenveroorde...,None,Overwegingen,1,Beoordeling
8458,ECLI:NL:RBZWB:2015:638.txt,De rechtbank verklaart het beroep ongegron,beslissing,Beslissing,1,Beslissing
8459,ECLI:NL:RBZWB:2015:638.txt,"Deze uitspraak is gedaan door [naam], voorzitt...",None,Beslissing,1,Beslissing


In [34]:
df = model_data.copy()

df["y_labelled"] = (df["label"] != "None").astype(int)

df["y_labelled"].value_counts()


y_labelled
1    5331
0    3060
Name: count, dtype: int64

In [35]:
df["y_labelled"].value_counts(normalize=True)


y_labelled
1    0.635324
0    0.364676
Name: proportion, dtype: float64

In [36]:
print("Total rows:", len(df))
print("Unique cases:", df["case_name"].nunique())
print(df.groupby("y_labelled").size())


Total rows: 8391
Unique cases: 100
y_labelled
0    3060
1    5331
dtype: int64


## Train Test Split

In [37]:
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


# ============================================================
# 1) TRAIN / EVAL / TEST SPLIT (case-level, one-time)
# ============================================================

# First split: train+eval vs test (80 / 20)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_eval_idx, test_idx = next(
    gss1.split(df, groups=df["case_name"])
)

train_eval_df = df.iloc[train_eval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Second split: train vs eval (75 / 25 of train_eval → 60 / 20 overall)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, eval_idx = next(
    gss2.split(train_eval_df, groups=train_eval_df["case_name"])
)

train_df = train_eval_df.iloc[train_idx].reset_index(drop=True)
eval_df  = train_eval_df.iloc[eval_idx].reset_index(drop=True)

# Quick sanity checks
print("Train:", len(train_df))
print("Eval :", len(eval_df))
print("Test :", len(test_df))

print("Labelled rate:",
      train_df["y_labelled"].mean().round(3),
      eval_df["y_labelled"].mean().round(3),
      test_df["y_labelled"].mean().round(3))

# ============================================================
from datasets import Dataset

# ============================================================
# 2) HF DATASETS (binary: text -> y_labelled)
# ============================================================

train_ds = Dataset.from_pandas(
    train_df[["sent_text", "y_labelled"]]
    .rename(columns={"sent_text": "text", "y_labelled": "labels"})
)

eval_ds = Dataset.from_pandas(
    eval_df[["sent_text", "y_labelled"]]
    .rename(columns={"sent_text": "text", "y_labelled": "labels"})
)

test_ds = Dataset.from_pandas(
    test_df[["sent_text", "y_labelled"]]
    .rename(columns={"sent_text": "text", "y_labelled": "labels"})
)

# ============================================================
# 3) MODEL DEFINITION & TOKENIZATION (shared preprocessing)
# ============================================================
model_name = "GroNLP/bert-base-dutch-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tok, batched=True)
eval_ds  = eval_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)




Train: 5608
Eval : 1281
Test : 1502
Labelled rate: 0.644 0.593 0.64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/5608 [00:00<?, ? examples/s]

Map:   0%|          | 0/1281 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

## Helper Function

In [38]:
import os, json
from datetime import datetime

best_models = {}  # global registry (in-memory; optionally saved to disk)

def register_best_model(
    registry: dict,
    model_key: str,
    trainer,
    tokenizer,
    base_save_dir: str = "models",
    extra_meta: dict | None = None,
):
    """
    Save best model + tokenizer from a HuggingFace Trainer run to a stable folder,
    and register metadata in `registry`.

    Parameters
    ----------
    registry : dict
        A dict that will store metadata for saved models (e.g., best_models).
    model_key : str
        Unique key for this model (e.g., "stage1_gatekeeper").
    trainer : transformers.Trainer
        Trainer instance used for training.
    tokenizer :
        Tokenizer used for training.
    base_save_dir : str
        Root directory where models will be saved.
    extra_meta : dict | None
        Optional metadata to store (stage, task, labels, dataset version, etc.)
    """
    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt is None:
        raise ValueError(
            "No best_model_checkpoint found. Did you set evaluation_strategy and load_best_model_at_end=True?"
        )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(base_save_dir, f"{model_key}_best_{timestamp}")
    os.makedirs(save_path, exist_ok=True)

    trainer.model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    meta = {
        "model_key": model_key,
        "saved_path": save_path,
        "best_checkpoint": best_ckpt,
        "time_saved": timestamp,
        "training_args": trainer.args.to_dict() if hasattr(trainer, "args") else None,
        "global_step": getattr(trainer.state, "global_step", None),
        "best_metric": getattr(trainer.state, "best_metric", None),
        "best_model_checkpoint": getattr(trainer.state, "best_model_checkpoint", None),
    }
    if extra_meta:
        meta.update(extra_meta)

    registry[model_key] = meta

    with open(os.path.join(save_path, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    return meta


train_df

In [39]:
train_df

,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled
0,ECLI:NL:GHARL:2016:7748.txt,Bij voormeld tussenarrest heeft het hof partij...,None,De verdere motivering van de beslissing in hog...,1,Beoordeling,0
1,ECLI:NL:GHARL:2016:7748.txt,Ter onderbouwing van zijn stelling dat er geen...,proceshandelingen,De verdere motivering van de beslissing in hog...,1,Beoordeling,1
2,ECLI:NL:GHARL:2016:7748.txt,Ook volgens [geïntimeerde] is het hof gebonden...,None,De verdere motivering van de beslissing in hog...,1,Beoordeling,0
3,ECLI:NL:GHARL:2016:7748.txt,Hetgeen [appellant] met verwijzing naar de als...,beoordeling,De verdere motivering van de beslissing in hog...,1,Beoordeling,1
4,ECLI:NL:GHARL:2016:7748.txt,Daaruit blijkt dat [appellant] zich jegens [ge...,beoordeling,De verdere motivering van de beslissing in hog...,1,Beoordeling,1
...,...,...,...,...,...,...,...
5603,ECLI:NL:RBZWB:2015:638.txt,Het bestreden besluit kan in stand blijven.,beoordeling,Overwegingen,1,Beoordeling,1
5604,ECLI:NL:RBZWB:2015:638.txt,Er is geen grond voor een proceskostenveroorde...,None,Overwegingen,1,Beoordeling,0
5605,ECLI:NL:RBZWB:2015:638.txt,De rechtbank verklaart het beroep ongegron,beslissing,Beslissing,1,Beslissing,1
5606,ECLI:NL:RBZWB:2015:638.txt,"Deze uitspraak is gedaan door [naam], voorzitt...",None,Beslissing,1,Beslissing,0


# Training 

## 1. Full 5 way classification

### Training

In [43]:
# ============================================================
# TRAIN TEXT-ONLY 5-WAY CLASSIFIER (RobBERT)
# Uses frozen splits: train_df / eval_df / test_df
# ============================================================

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

# ------------------------------------------------------------
# 1) Config + label mapping (fixed order!)
# ------------------------------------------------------------
MODEL_NAME = "GroNLP/bert-base-dutch-cased"

LEGAL_LABELS = ["None", "beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
label2id = {l: i for i, l in enumerate(LEGAL_LABELS)}
id2label = {i: l for l, i in label2id.items()}

train_legal = train_df.copy()
eval_legal  = eval_df.copy()
test_legal  = test_df.copy()   # keep untouched; evaluate later

train_legal["y"] = train_legal["label"].map(label2id)
eval_legal["y"]  = eval_legal["label"].map(label2id)
test_legal["y"]  = test_legal["label"].map(label2id)

# ------------------------------------------------------------
# 3) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------
# 4) Torch Dataset wrapper (text -> tokenized tensors)
# ------------------------------------------------------------
class TextDataset(Dataset):
    def __init__(self, df, text_col="sent_text", label_col="y", tokenizer=None, max_len=256):
        self.texts = df[text_col].astype(str).tolist()
        self.labels = df[label_col].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds_5 = TextDataset(train_legal, tokenizer=tokenizer, max_len=256)
eval_ds_5  = TextDataset(eval_legal,  tokenizer=tokenizer, max_len=256)
test_ds_5  = TextDataset(test_legal,  tokenizer=tokenizer, max_len=256)

# ------------------------------------------------------------
# 5) Model
# ------------------------------------------------------------
model_5 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LEGAL_LABELS),
    id2label=id2label,
    label2id=label2id
)

# ------------------------------------------------------------
# 6) Metrics (5-way)
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "micro_f1": f1_score(labels, preds, average="micro"),
    }

# ------------------------------------------------------------
# 7) Training arguments
# ------------------------------------------------------------
training_args_5 = TrainingArguments(
    output_dir="./text_only_model_5",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    fp16=True
)

# ------------------------------------------------------------
# 8) Trainer (IMPORTANT: eval_dataset = eval_ds_4, not test)
# ------------------------------------------------------------
trainer_5 = Trainer(
    model=model_5,
    args=training_args_5,
    train_dataset=train_ds_5,
    eval_dataset=eval_ds_5,          #validation split
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 9) Train Stage 2A
# ------------------------------------------------------------
trainer_5.train()

# ------------------------------------------------------------
# 10) Save best Stage 2A model into registry
# ------------------------------------------------------------

os.makedirs("models", exist_ok=True)

if "best_models" not in globals():
    best_models = {}

stage2a_meta = register_best_model(
    registry=best_models,
    model_key="text_only_5way",
    trainer=trainer_5,
    tokenizer=tokenizer,
    base_save_dir="models",
    extra_meta={
        "stage": 0,
        "experiment": "1",
        "variant": "text_only",
        "task": "5_way_legal_role_classification",
        "model_name": MODEL_NAME,
        "labels_order": LEGAL_LABELS,
        "text_col": "sent_text",
        "max_len": 256,
        "train_rows": len(train_legal),
        "eval_rows": len(eval_legal),
        "test_rows": len(test_legal),
    }
)

print("✅ Saved 5way best model to:", stage2a_meta["saved_path"])
print("Registry keys now:", list(best_models.keys()))

# Persist registry snapshot to disk (single source of truth)
with open("models/best_models_registry.json", "w", encoding="utf-8") as f:
    json.dump(best_models, f, indent=2, ensure_ascii=False)


import torch
import gc

del model_5
del trainer_5 # if using Hugging Face
gc.collect()
torch.cuda.empty_cache() 

# Now the next model can start with a fresh 8GB





Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\P70092814\AppData\Local\Temp\ipykernel_27036\135583952.py:107: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_5 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Micro F1
1,0.917800,0.926390,0.608899,0.639803,0.608899
2,0.615100,0.903207,0.637002,0.661375,0.637002
3,0.435200,1.007731,0.646370,0.675475,0.646370
4,0.260300,1.126701,0.632319,0.662314,0.632319


✅ Saved 5way best model to: models\text_only_5way_best_20260506_122247
Registry keys now: ['text_only_5way']


In [41]:
best_models

{'text_only_5way': {'model_key': 'text_only_5way',
  'saved_path': 'models\\text_only_5way_best_20260506_120157',
  'best_checkpoint': './text_only_model_5\\checkpoint-702',
  'time_saved': '20260506_120157',
  'training_args': {'output_dir': './text_only_model_5',
   'overwrite_output_dir': False,
   'do_train': False,
   'do_eval': True,
   'do_predict': False,
   'eval_strategy': 'epoch',
   'prediction_loss_only': False,
   'per_device_train_batch_size': 16,
   'per_device_eval_batch_size': 32,
   'per_gpu_train_batch_size': None,
   'per_gpu_eval_batch_size': None,
   'gradient_accumulation_steps': 1,
   'eval_accumulation_steps': None,
   'eval_delay': 0,
   'torch_empty_cache_steps': None,
   'learning_rate': 2e-05,
   'weight_decay': 0.01,
   'adam_beta1': 0.9,
   'adam_beta2': 0.999,
   'adam_epsilon': 1e-08,
   'max_grad_norm': 1.0,
   'num_train_epochs': 4,
   'max_steps': -1,
   'lr_scheduler_type': 'linear',
   'lr_scheduler_kwargs': {},
   'warmup_ratio': 0.0,
   'warmup_

## 2. Pipeline Stage 1

In [46]:

import torch
import gc

gc.collect()
torch.cuda.empty_cache() 

In [47]:
# ============================================================
# STAGE 1 — TRAINING (Binary Gatekeeper: None vs Labelled)
# ================================a============================

from transformers import TrainingArguments
# ============================================================
# 3) MODEL DEFINITION & TOKENIZATION (shared preprocessing)
# ============================================================
model_name = "GroNLP/bert-base-dutch-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ------------------------------------------------------------
# 1) Training configuration
# ------------------------------------------------------------
args = TrainingArguments(
    output_dir="robbert_gatekeeper",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,

    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,  # use mixed precision if available
    report_to="none",
)

# ------------------------------------------------------------
# 2) Metrics definition
# ------------------------------------------------------------
import evaluate
import numpy as np

metric_f1 = evaluate.load("f1")
metric_pr = evaluate.load("precision")
metric_rc = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": metric_pr.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": metric_rc.compute(predictions=preds, references=labels, average="binary")["recall"],
    }

# ------------------------------------------------------------
# 3) Trainer definition (uses frozen train/eval splits)
# ------------------------------------------------------------
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,     
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------
# 4) Train Stage 1 model
# ------------------------------------------------------------
trainer.train()
stage1_meta = register_best_model(
    registry=best_models,
    model_key="stage1_gatekeeper",
    trainer=trainer,
    tokenizer=tokenizer,
    extra_meta={
        "stage": 1,
        "task": "binary_labelled_vs_unlabelled",
        "labels": ["not_labelled", "labelled"],
    }
)

print("Saved + registered:", stage1_meta["saved_path"])
print("Registry keys:", list(best_models.keys()))

os.makedirs("models", exist_ok=True)

with open("models/best_models_registry.json", "w", encoding="utf-8") as f:
    json.dump(best_models, f, indent=2, ensure_ascii=False)


import torch
import gc

del model
del trainer # if using Hugging Face
gc.collect()
torch.cuda.empty_cache() 

# Now the next model can start with a fresh 8GB


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,No log,0.544452,0.796957,0.716842,0.897233
2,0.452900,0.636810,0.793609,0.802153,0.785244
3,0.206300,0.833066,0.814115,0.780193,0.851120
4,0.206300,1.273486,0.818465,0.777251,0.864295
5,0.077200,1.477268,0.812068,0.776442,0.851120
6,0.034000,1.529224,0.813939,0.771226,0.861660


Saved + registered: models\stage1_gatekeeper_best_20260506_140445
Registry keys: ['text_only_5way', 'stage1_gatekeeper']


## 3. Pipeline (Stage 2)
   4-way Model &
   Individual Models

In [ ]:
# ============================================================
# STAGE 2 A — TRAIN TEXT-ONLY 4-WAY CLASSIFIER (RobBERT)
# Uses frozen splits: train_legal / eval_legal / test_legal
# ============================================================

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

# ------------------------------------------------------------
# 1) Config + label mapping (fixed order!)
# ------------------------------------------------------------
MODEL_NAME = "GroNLP/bert-base-dutch-cased"

LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
label2id = {l: i for i, l in enumerate(LEGAL_LABELS)}
id2label = {i: l for l, i in label2id.items()}

# 2) Start from the frozen splits (from Stage 1 split)
train_df_all = train_df.copy()
eval_df_all  = eval_df.copy()
test_df_all  = test_df.copy()

train_legal = train_df_all[train_df_all["label"].isin(LEGAL_LABELS)].copy()
eval_legal  = eval_df_all[eval_df_all["label"].isin(LEGAL_LABELS)].copy()
test_legal  = test_df_all[test_df_all["label"].isin(LEGAL_LABELS)].copy()

train_legal = train_legal.copy()
eval_legal  = eval_legal.copy()
test_legal  = test_legal.copy()   # keep untouched; evaluate later

train_legal["y"] = train_legal["label"].map(label2id)
eval_legal["y"]  = eval_legal["label"].map(label2id)
test_legal["y"]  = test_legal["label"].map(label2id)

# ------------------------------------------------------------
# 3) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------
# 4) Torch Dataset wrapper (text -> tokenized tensors)
# ------------------------------------------------------------
class TextDataset(Dataset):
    def __init__(self, df, text_col="sent_text", label_col="y", tokenizer=None, max_len=256):
        self.texts = df[text_col].astype(str).tolist()
        self.labels = df[label_col].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds_4 = TextDataset(train_legal, tokenizer=tokenizer, max_len=256)
eval_ds_4  = TextDataset(eval_legal,  tokenizer=tokenizer, max_len=256)
test_ds_4  = TextDataset(test_legal,  tokenizer=tokenizer, max_len=256)

# ------------------------------------------------------------
# 5) Model
# ------------------------------------------------------------
model_4 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LEGAL_LABELS),
    id2label=id2label,
    label2id=label2id
)

# ------------------------------------------------------------
# 6) Metrics (4-way)
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "micro_f1": f1_score(labels, preds, average="micro"),
    }

# ------------------------------------------------------------
# 7) Training arguments
# ------------------------------------------------------------
training_args_4 = TrainingArguments(
    output_dir="./text_only_model_4",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    report_to="none"
)

# ------------------------------------------------------------
# 8) Trainer (IMPORTANT: eval_dataset = eval_ds_4, not test)
# ------------------------------------------------------------
trainer_4 = Trainer(
    model=model_4,
    args=training_args_4,
    train_dataset=train_ds_4,
    eval_dataset=eval_ds_4,          #validation split
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 9) Train Stage 2A
# ------------------------------------------------------------
trainer_4.train()

# ------------------------------------------------------------
# 10) Save best Stage 2A model into registry
# ------------------------------------------------------------

os.makedirs("models", exist_ok=True)

if "best_models" not in globals():
    best_models = {}

stage2a_meta = register_best_model(
    registry=best_models,
    model_key="stage2_text_only_4way",
    trainer=trainer_4,
    tokenizer=tokenizer,
    base_save_dir="models",
    extra_meta={
        "stage": 2,
        "experiment": "2A",
        "variant": "text_only",
        "task": "4_way_legal_role_classification",
        "model_name": MODEL_NAME,
        "labels_order": LEGAL_LABELS,
        "text_col": "sent_text",
        "max_len": 256,
        "train_rows": len(train_legal),
        "eval_rows": len(eval_legal),
        "test_rows": len(test_legal),
        
    }
)

print("✅ Saved Stage 2A best model to:", stage2a_meta["saved_path"])
print("Registry keys now:", list(best_models.keys()))

# Persist registry snapshot to disk (single source of truth)
with open("models/best_models_registry.json", "w", encoding="utf-8") as f:
    json.dump(best_models, f, indent=2, ensure_ascii=False)


import torch
import gc

gc.collect()
torch.cuda.empty_cache() 


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\P70092814\AppData\Local\Temp\ipykernel_27036\1491212871.py:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_4 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Micro F1
1,No log,0.604420,0.764163,0.780879,0.764163
2,No log,0.619826,0.765481,0.788262,0.765481
3,0.605200,0.696668,0.754941,0.783328,0.754941
4,0.605200,0.752862,0.765481,0.792971,0.765481


✅ Saved Stage 2A best model to: models\stage2_text_only_4way_best_20260506_141904
Registry keys now: ['text_only_5way', 'stage1_gatekeeper', 'stage2_text_only_4way']


NameError: name 'model' is not defined

In [51]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache() 

In [52]:
# ============================================================
# STAGE 2 B — TRAIN 4 OVRS (One-vs-Rest) CLASSIFIERS (RobBERT)
# Uses frozen splits: train_legal / eval_legal / test_legal
# ============================================================
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import classification_report, f1_score, accuracy_score
import evaluate

MODEL_NAME = "GroNLP/bert-base-dutch-cased"
LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]

tokenizer_ovr = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(batch):
    return tokenizer_ovr(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# Make OVR dataframe
def make_ovr_df(df_legal: pd.DataFrame, pos_label: str) -> pd.DataFrame:
    """
    Option 2.1:
      - df_legal contains ONLY rows with label in LEGAL_LABELS
      - y = 1 if label == pos_label else 0
    """
    out = df_legal.copy()
    out["y"] = (out["label"] == pos_label).astype(int)
    return out

def make_hf_ds(df_bin: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        df_bin[["sent_text", "y"]].rename(columns={"sent_text": "text", "y": "labels"})
    )
    ds = ds.map(tok, batched=True)
    return ds

#Train one OVR model per label
metric_f1 = evaluate.load("f1")
metric_pr = evaluate.load("precision")
metric_rc = evaluate.load("recall")

def compute_metrics_bin(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": metric_pr.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": metric_rc.compute(predictions=preds, references=labels, average="binary")["recall"],
    }

def train_one_ovr_model(pos_label: str, train_legal: pd.DataFrame, eval_legal: pd.DataFrame):
    # Build binary train/eval
    train_bin_df = make_ovr_df(train_legal, pos_label)
    eval_bin_df  = make_ovr_df(eval_legal, pos_label)

    train_ds = make_hf_ds(train_bin_df)
    eval_ds  = make_hf_ds(eval_bin_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./ovr_{pos_label.replace(' ', '_')}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        fp16=True,
        
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics_bin,
        tokenizer=tokenizer_ovr,
    )

    trainer.train()
    return trainer


ovr_trainers = {}

ovr_trainers = {}

for lab in LEGAL_LABELS[2:]:  # just run one for demo
    gc.collect()
    torch.cuda.empty_cache() 
    print(f"\n==============================")
    print(f"Training OvR model for: {lab}")
    print(f"==============================")

    import torch

    print("CUDA available:", torch.cuda.is_available())
    print("CUDA device count:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
    

    tr = train_one_ovr_model(lab, train_legal, eval_legal)
    ovr_trainers[lab] = tr

    # ---- Register best model in global registry ----
    model_key = f"ovr_{lab.replace(' ', '_')}".lower()

    meta = register_best_model(
        registry=best_models,
        model_key=model_key,
        trainer=tr,
        tokenizer=tokenizer_ovr,   
        extra_meta={
            "stage": 2,
            "type": "ovr_binary",
            "pos_label": lab,
            "labels": ["not_"+lab, lab],  
        }
    )

    print("Saved:", meta["saved_path"])
    print("Best ckpt:", meta["best_checkpoint"])
    print("Registry keys:", list(best_models.keys()))

    os.makedirs("models", exist_ok=True)

    with open("models/best_models_registry.json", "w", encoding="utf-8") as f:
        json.dump(best_models, f, indent=2, ensure_ascii=False)





Training OvR model for: materiele feiten
CUDA available: True
CUDA device count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


Map:   0%|          | 0/3611 [00:00<?, ? examples/s]

Map:   0%|          | 0/759 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\P70092814\AppData\Local\Temp\ipykernel_27036\3721775749.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,No log,0.350818,0.690217,0.841060,0.585253
2,No log,0.391372,0.694175,0.733333,0.658986
3,0.292100,0.490669,0.706173,0.760638,0.658986


...

Map:   0%|          | 0/3611 [00:00<?, ? examples/s]

Map:   0%|          | 0/759 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\P70092814\AppData\Local\Temp\ipykernel_27036\3721775749.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,No log,0.414477,0.568562,0.752212,0.456989
2,No log,0.472942,0.578778,0.720000,0.483871
3,0.329300,0.562523,0.609231,0.712230,0.532258


Saved: models\ovr_proceshandelingen_best_20260506_160617
Best ckpt: ./ovr_proceshandelingen\checkpoint-678
Registry keys: ['text_only_5way', 'stage1_gatekeeper', 'stage2_text_only_4way', 'ovr_beoordeling', 'ovr_beslissing', 'ovr_materiele_feiten', 'ovr_proceshandelingen']
